# Upload to database: products

This notebook will outline the process of loading the csv files with the products and uploading it to the database using PostgreSQL and SQLAlchemy within Python

In [69]:
# Imports

import pandas as pd 
from dotenv import load_dotenv, dotenv_values
import os

from sqlalchemy import create_engine, types
from sqlalchemy import text # to be able to pass string
from sqlalchemy import Integer, String, Float, DateTime, Date


In [70]:
# Loading values from .env
config = dotenv_values()

# Define variables for the login
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
dbname = os.getenv("DB_NAME")
schema = os.getenv("DB_SCHEMA")

# PostgreSQL URL creation
url = f'postgresql://{user}:{password}@{host}:{port}/{dbname}'


In [71]:
# Create the engine to load data
engine = create_engine(url, echo=False)

# Setting schema
with engine.begin() as conn: 
    result = conn.execute(text(f'SET search_path TO {schema};'))

In [72]:
# Load table of sales
with engine.connect() as conn:
    conn.execute(text(f"SET search_path TO {schema}"))
    products = pd.read_sql(
        text("SELECT * FROM jl_sales"),
        conn
    )


In [73]:
# We will check here the columns and types

products

,Sales Person,Country,Product,Date,Amount,Boxes Shipped,Year,Month
0,Jehu Rudeforth,UK,Mint Chip Choco,2022-01-04,5320.00,180,2022,1
1,Van Tuxwell,India,85% Dark Bars,2022-08-01,7896.00,94,2022,8
2,Gigi Bohling,India,Peanut Butter Cubes,2022-07-07,4501.00,91,2022,7
3,Jan Morforth,Australia,Peanut Butter Cubes,2022-04-27,12726.00,342,2022,4
4,Jehu Rudeforth,UK,Peanut Butter Cubes,2022-02-24,13685.00,184,2022,2
...,...,...,...,...,...,...,...,...
3277,Karlen McCaffrey,Australia,Spicy Special Slims,2024-05-17,5303.58,354,2024,5
3278,Jehu Rudeforth,USA,White Choc,2024-06-07,7339.32,121,2024,6
3279,Ches Bonnell,Canada,Organic Choco Syrup,2024-07-26,616.09,238,2024,7
3280,Dotty Strutley,India,Eclairs,2024-07-28,2504.62,397,2024,7


In [74]:
df_products = pd.DataFrame(products['Product'].unique(), columns=['Product'])

df_products

,Product
0,Mint Chip Choco
1,85% Dark Bars
2,Peanut Butter Cubes
3,Smooth Sliky Salty
4,99% Dark & Pure
5,After Nines
6,50% Dark Bites
7,Orange Choco
8,Eclairs
9,Drinking Coco


In [75]:
# Filling with zeros a second column
df_products['Quantity'] = ("0")

# Unique values
df_products = df_products.drop_duplicates()
df_products

,Product,Quantity
0,Mint Chip Choco,0
1,85% Dark Bars,0
2,Peanut Butter Cubes,0
3,Smooth Sliky Salty,0
4,99% Dark & Pure,0
5,After Nines,0
6,50% Dark Bites,0
7,Orange Choco,0
8,Eclairs,0
9,Drinking Coco,0


In [76]:
# Upload the dataframe to the database

df_products.to_sql(
    "jl_products_list",
    engine,
    schema = schema,
    if_exists="replace", # this replaces an existing table!
    index=False,
    dtype={
        "Product": String(),
        "Ponder": Float()
    }
)

22